# 🤖 AI Debator Project
This notebook creates a debate simulation between two AI personalities (Johny and Maria) using Google's Gemini API and a Gradio interface.

## Import required libraries
We import Python libraries for environment handling and Google Gemini client.

In [ ]:
import os
from dotenv import load_dotenv   # For loading environment variables
from google import genai         # Gemini AI client library

## Load API Key
We load the Gemini API key from the `.env` file and check if it's available.

In [ ]:
load_dotenv(override=True)  # Load environment variables from .env file

# Get Gemini API key
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Gemini API Key is available")
else:
    print("Gemini API Key is not available")

## Initialize Gemini client
We create a client object using the API key.

In [ ]:
# Create the Gemini client using the API key
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

## Define Johny's personality function
Johny is a calm and cool debator. He responds with short, simple sentences.

In [ ]:
def message_johny(input):
    print("Input to Johny:", input)
    
    # Request Gemini model to generate content in streaming mode
    response_stream = client.models.generate_content_stream(
        model="gemini-2.5-flash",
        contents="""You are a cool and calm person.
        You will be given a topic or a conversation between 2 persons.
        You have to share your thoughts as a next conversation.
        Reply only the next thought and not the entire conversation.
        Reply in markdown and in 1 sentence of 10 words with simple vocabulary.
        Conversation : \n """ + str(input)
    )

    result = ""
    # Collect streamed responses from Gemini
    for chunk in response_stream:
        result += chunk.text
        yield chunk.text

## Define Maria's personality function
Maria is angry and aggressive in her debate style.

In [ ]:
def message_maria(input):
    print("Input to Maria:", input)
    
    # Request Gemini model to generate content in streaming mode
    response_stream = client.models.generate_content_stream(
        model="gemini-2.5-flash",
        contents="""You are an angry person.
        You will be given a topic or a conversation between 2 persons.
        You have to share your thoughts as a next conversation.
        Reply only the next thought and not the entire conversation.
        Reply in markdown and in 1 sentence of 10 words with simple vocabulary.
        Conversation : \n """ + str(input)
    )

    result = ""
    # Collect streamed responses from Gemini
    for chunk in response_stream:
        result += chunk.text
        yield chunk.text

## Debate function
This function alternates between Johny and Maria, simulating a back-and-forth debate for 10 turns.

In [ ]:
def message_gemini(topic):
    johny_trun = True   # Flag to alternate turns

    text_on_ui = f"Debate Started on:\n {topic} \n\n"

    # Run for 10 turns
    for i in range(10):
        if johny_trun:
            text_on_ui += "## Johny Says:\n\n"
            for response in message_johny(text_on_ui):
                text_on_ui += response
                yield text_on_ui
                text_on_ui += "\n\n"
            johny_trun = False
        else:
            text_on_ui += "## Maria Says:\n\n"
            for response in message_maria(text_on_ui):
                text_on_ui += response
                yield text_on_ui
                text_on_ui += "\n\n"
            johny_trun = True

## Build Gradio UI
We use Gradio to create a simple interface where the user enters a topic and sees the debate unfold.

In [ ]:
import gradio as gr

# Gradio interface setup
view = gr.Interface(
    fn=message_gemini,                        # Function to run debate
    inputs=[gr.Textbox(label="Your message:")],  # Input box for debate topic
    outputs=[gr.Markdown(label="Response:")],    # Output shows debate conversation
    flagging_mode="never"                    # Disable flagging
)

# Launch the UI
view.launch()